In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("Pertemuan6-ParquetETL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/24 13:16:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [2]:
# Membuat dataset yang lebih besar (20.000 baris) khusus untuk latihan ini
np.random.seed(11)
n = 20000
data_besar = {
    "order_id": [f"O{i}" for i in range(n)],
    "customer_id": np.random.randint(1, 21, size=n),
    "kategori": np.random.choice(["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"], size=n),
    "kota": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"], size=n),
    "metode_pembayaran": np.random.choice(["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"], size=n),
    "pendapatan": np.random.randint(50000, 500000, size=n),
}
pd.DataFrame(data_besar).to_csv("data_transaksi_besar.csv", index=False)

csv_size = os.path.getsize("data_transaksi_besar.csv")
print(f"Ukuran CSV: {csv_size:,} bytes ({csv_size/1024:.1f} KB) untuk {n:,} baris")

Ukuran CSV: 892,909 bytes (872.0 KB) untuk 20,000 baris


In [5]:
# Membaca CSV, lalu menuliskannya ulang sebagai Parquet
df_besar = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_besar.write.mode("overwrite").parquet("data_transaksi_besar_parquet")

# Menghitung total ukuran seluruh berkas .parquet yang dihasilkan
parquet_size = sum(
    os.path.getsize(os.path.join(dp, f))
    for dp, dn, fn in os.walk("data_transaksi_besar_parquet")
    for f in fn if f.endswith(".parquet")
)

print(f"Ukuran CSV      : {csv_size:,} bytes ({csv_size/1024:.1f} KB)")
print(f"Ukuran Parquet  : {parquet_size:,} bytes ({parquet_size/1024:.1f} KB)")
print(f"Parquet {(1 - parquet_size/csv_size)*100:.1f}% lebih kecil dibanding CSV")

[Stage 4:>                                                          (0 + 1) / 1]

Ukuran CSV      : 892,909 bytes (872.0 KB)
Ukuran Parquet  : 202,197 bytes (197.5 KB)
Parquet 77.4% lebih kecil dibanding CSV


In [3]:
import json

# Membuat data pelanggan sebagai JSON Lines (satu objek JSON per baris — format umum untuk big data)
np.random.seed(21)
pelanggan = [
    {
        "customer_id": i,
        "nama": f"Pelanggan{i}",
        "membership": np.random.choice(["Silver", "Gold", "Platinum"]),
        "kota_domisili": np.random.choice(["Magelang", "Semarang", "Solo", "Yogyakarta", "Purworejo"])
    }
    for i in range(1, 21)
]

with open("data_pelanggan.json", "w") as f:
    for p in pelanggan:
        f.write(json.dumps(p) + "\n")

print("Berkas data_pelanggan.json berhasil dibuat (format JSON Lines).")

Berkas data_pelanggan.json berhasil dibuat (format JSON Lines).


In [4]:
# Membaca JSON menggunakan Spark — Spark otomatis mendeteksi struktur/skema JSON
df_pelanggan = spark.read.json("data_pelanggan.json")
df_pelanggan.printSchema()
df_pelanggan.show(5)

root
 |-- customer_id: long (nullable = true)
 |-- kota_domisili: string (nullable = true)
 |-- membership: string (nullable = true)
 |-- nama: string (nullable = true)

+-----------+-------------+----------+----------+
|customer_id|kota_domisili|membership|      nama|
+-----------+-------------+----------+----------+
|          1|     Magelang|      Gold|Pelanggan1|
|          2|     Magelang|    Silver|Pelanggan2|
|          3|     Magelang|    Silver|Pelanggan3|
|          4|         Solo|    Silver|Pelanggan4|
|          5|         Solo|      Gold|Pelanggan5|
+-----------+-------------+----------+----------+
only showing top 5 rows



In [6]:
# Persiapan ulang untuk latihan
spark = SparkSession.builder.appName("Latihan6").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_transaksi_etl = spark.read.csv("data_transaksi_besar.csv", header=True, inferSchema=True)
df_pelanggan_etl = spark.read.json("data_pelanggan.json")
print("Siap untuk latihan.")

Siap untuk latihan.


In [7]:
# Baca kembali data_transaksi_besar_parquet menggunakan spark.read.parquet()
df_transaksi_besar_parquet = spark.read.parquet("data_transaksi_besar_parquet")

print("Skema hasil spark.read.parquet():")
df_transaksi_besar_parquet.printSchema()

print("Skema hasil inferSchema=True pada CSV (df_transaksi_etl):")
df_transaksi_etl.printSchema()

#print("Kesimpulan: Skema hasil spark.read.parquet() dan hasil inferSchema=True pada CSV adalah SAMA PERSIS.")
#print("Kedua sumber menghasilkan tipe data yang identik untuk setiap kolom (string, integer).")
#print("Ini menunjukkan bahwa untuk dataset ini, inferSchema=True pada CSV cukup akurat menebak tipe data,")
#print("kemungkinan karena format datanya konsisten dan tidak ada nilai yang ambigu/kosong.")

Skema hasil spark.read.parquet():
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)

Skema hasil inferSchema=True pada CSV (df_transaksi_etl):
root
 |-- order_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- pendapatan: integer (nullable = true)



In [10]:
# Jawaban Soal 2 di sini
df_gabungan = df_transaksi_etl.join(df_pelanggan_etl, on="customer_id", how="inner")

df_gabungan.groupBy("membership").count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum| 4016|
|    Silver| 8994|
|      Gold| 6990|
+----------+-----+



In [11]:
# Jawaban Soal 3 di sini
df_gabungan.write.mode("overwrite").partitionBy("membership").parquet("hasil_gabungan_parquet")

!ls -R hasil_gabungan_parquet

hasil_gabungan_parquet:
'membership=Gold'  'membership=Platinum'  'membership=Silver'   _SUCCESS

'hasil_gabungan_parquet/membership=Gold':
part-00000-1b61a6b7-98e4-4003-b07c-3966de52019e.c000.snappy.parquet

'hasil_gabungan_parquet/membership=Platinum':
part-00000-1b61a6b7-98e4-4003-b07c-3966de52019e.c000.snappy.parquet

'hasil_gabungan_parquet/membership=Silver':
part-00000-1b61a6b7-98e4-4003-b07c-3966de52019e.c000.snappy.parquet


In [12]:
#Partitioning berdasarkan kolom `kota` cocok karena jumlah nilai uniknya sedikit (hanya beberapa kota), sehingga data terbagi ke sejumlah folder yang wajar dan query yang memfilter berdasarkan kota bisa langsung skip ke folder yang relevan (partition pruning), sehingga lebih efisien. Sebaliknya, jika partitioning diterapkan pada kolom seperti `order_id` yang nilainya unik di setiap baris, maka akan tercipta folder terpisah untuk setiap baris data, menghasilkan ribuan/jutaan folder kecil (small file problem) yang justru memperlambat proses baca-tulis dan membebani metadata filesystem, bukannya mempercepat query.

In [13]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
